# 02. Exploratory Data Analysis and Visualization

This notebook analyzes the cleaned dataset and visualizes the fraud patterns that are present in the actual schema.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(Path.cwd().parent / 'data' / 'processed' / 'cleaned_upi_dataset.csv')
target_col = [c for c in df.columns if 'fraud' in c.lower() or 'label' in c.lower() or 'target' in c.lower()][0]
print('Target column:', target_col)
df[target_col].value_counts()

In [ ]:
counts = df[target_col].value_counts().rename({0: 'Genuine', 1: 'Fraud'})
sns.barplot(x=counts.index, y=counts.values, palette=['#3b5b92', '#d97706'])
plt.title('Genuine vs Fraudulent Transactions')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()

In [ ]:
amount_col = next((c for c in df.columns if 'amount' in c.lower() or 'value' in c.lower()), None)
if amount_col is not None:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[amount_col], bins=40, kde=True, color='#3b5b92')
    plt.title('Transaction Amount Distribution')
    plt.xlabel('Amount')
    plt.ylabel('Frequency')
    plt.tight_layout()

    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x=target_col, y=amount_col, palette=['#3b5b92', '#d97706'])
    plt.title('Transaction Amount by Fraud Status')
    plt.xlabel('Class')
    plt.ylabel('Amount')
    plt.tight_layout()

In [ ]:
for col in [c for c in df.columns if any(token in c.lower() for token in ['type', 'category', 'device', 'network', 'bank', 'state'])]:
    if df[col].nunique() <= 20:
        rates = df.groupby(col)[target_col].mean().sort_values(ascending=False)
        plt.figure(figsize=(9, 4))
        sns.barplot(x=rates.index, y=rates.values, palette='deep')
        plt.title(f'Fraud Rate by {col}')
        plt.xticks(rotation=25, ha='right')
        plt.ylabel('Fraud rate')
        plt.tight_layout()
        break

In [ ]:
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()